In [ ]:
# Import required libraries
import sys
sys.path.append('..')

import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
%matplotlib inline

from src.extractor import AudioExtractor, midi_number_to_note_name
from src import utils

## 1. Generate Test Audio

Create a synthetic melody to test pitch detection accuracy.

In [ ]:
# Generate test melody: C major scale
sr = 22050
duration = 0.5  # seconds per note

# MIDI notes for C major scale
midi_notes = [60, 62, 64, 65, 67, 69, 71, 72]  # C4 to C5
frequencies = [librosa.midi_to_hz(m) for m in midi_notes]

# Generate sine waves
audio = np.array([])
for freq in frequencies:
    t = np.linspace(0, duration, int(sr * duration))
    note = 0.5 * np.sin(2 * np.pi * freq * t)
    # Add fade in/out
    fade_len = int(sr * 0.05)
    note[:fade_len] *= np.linspace(0, 1, fade_len)
    note[-fade_len:] *= np.linspace(1, 0, fade_len)
    audio = np.concatenate([audio, note])

print(f"Generated {len(audio)} samples ({len(audio)/sr:.2f} seconds)")
print(f"Notes: {[midi_number_to_note_name(m) for m in midi_notes]}")

## 2. Visualize Audio Waveform

In [ ]:
plt.figure(figsize=(14, 4))
librosa.display.waveshow(audio, sr=sr)
plt.title('Test Melody Waveform')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

## 3. Compute Spectrogram

In [ ]:
# Compute Short-Time Fourier Transform
D = librosa.stft(audio)
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

plt.figure(figsize=(14, 6))
librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='hz')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram')
plt.ylim([0, 2000])  # Focus on relevant frequency range
plt.tight_layout()
plt.show()

## 4. Test pYIN Pitch Detection

In [ ]:
# Save test audio
import soundfile as sf
test_path = '../data/raw/test_scale.wav'
sf.write(test_path, audio, sr)

# Extract with pYIN
extractor_pyin = AudioExtractor(method='pyin', min_note_duration=0.1)
notes_pyin = extractor_pyin.extract(test_path)

print(f"\nDetected {len(notes_pyin)} notes:")
for i, note in enumerate(notes_pyin):
    note_name = midi_number_to_note_name(note.midi_number)
    print(f"{i+1}. {note_name} (MIDI {note.midi_number}) - {note.onset:.2f}s to {note.offset:.2f}s")

## 5. Visualize Detected Pitches

In [ ]:
# Extract raw F0 for visualization
f0, voiced_flag, voiced_probs = librosa.pyin(
    audio,
    fmin=librosa.note_to_hz('C2'),
    fmax=librosa.note_to_hz('C6'),
    sr=sr
)

# Convert to MIDI
f0_midi = librosa.hz_to_midi(f0)

# Create time array
times = librosa.times_like(f0, sr=sr)

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: F0 in Hz
ax1.plot(times, f0, linewidth=2, label='Estimated F0')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Frequency (Hz)')
ax1.set_title('pYIN Pitch Detection')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: MIDI notes
ax2.plot(times, f0_midi, linewidth=2, label='Estimated MIDI')
# Overlay detected notes
for note in notes_pyin:
    ax2.axhspan(note.midi_number - 0.5, note.midi_number + 0.5,
                xmin=note.onset/times[-1], xmax=note.offset/times[-1],
                alpha=0.3, color='green', label='Detected' if note == notes_pyin[0] else '')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('MIDI Note Number')
ax2.set_title('MIDI Note Detection')
ax2.set_yticks(range(59, 74))
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Accuracy Evaluation

In [ ]:
# Compare detected notes with ground truth
ground_truth = midi_notes
detected = [n.midi_number for n in notes_pyin]

print("Accuracy Evaluation:")
print("=" * 60)
print(f"Ground truth notes: {len(ground_truth)}")
print(f"Detected notes:     {len(detected)}")

# Calculate accuracy
if len(detected) == len(ground_truth):
    correct = sum(1 for d, g in zip(detected, ground_truth) if d == g)
    accuracy = correct / len(ground_truth) * 100
    print(f"Correct notes:      {correct}/{len(ground_truth)}")
    print(f"Accuracy:           {accuracy:.1f}%")
    
    # Show errors
    print("\nDetailed comparison:")
    for i, (g, d) in enumerate(zip(ground_truth, detected)):
        status = "✓" if g == d else "✗"
        g_name = midi_number_to_note_name(g)
        d_name = midi_number_to_note_name(d)
        print(f"{status} Note {i+1}: Expected {g_name} (MIDI {g}), Got {d_name} (MIDI {d})")
else:
    print("WARNING: Number of detected notes doesn't match ground truth!")
    print(f"Expected: {[midi_number_to_note_name(m) for m in ground_truth]}")
    print(f"Detected: {[midi_number_to_note_name(m) for m in detected]}")

## 7. Test with Real Audio (Optional)

If you have a real vocal recording, test it here.

In [ ]:
# Uncomment and modify path to test with real audio
# real_audio_path = '../data/raw/vocal_sample.mp3'
# 
# if os.path.exists(real_audio_path):
#     # Analyze pitch range
#     pitch_info = utils.analyze_pitch_range(real_audio_path)
#     print("Pitch Analysis:")
#     for key, value in pitch_info.items():
#         print(f"  {key}: {value}")
#     
#     # Extract notes
#     notes = extractor_pyin.extract(real_audio_path)
#     print(f"\nExtracted {len(notes)} notes")
# else:
#     print(f"Audio file not found: {real_audio_path}")

print("Uncomment the code above to test with real audio")

## Summary

This notebook demonstrated:
1. Generating synthetic test audio
2. Visualizing waveforms and spectrograms
3. Testing pYIN pitch detection
4. Evaluating accuracy against ground truth

**Next Steps:**
- Try with real vocal recordings
- Test Basic Pitch method for comparison
- Tune parameters for better accuracy